<a id="sa-overview"></a>

$$
\text{模拟退火 (Simulated Annealing, SA): 模仿金属退火的随机优化算法} \\ \quad \\
\text{物理背景: 金属加热到高温, 原子可以自由移动; 慢慢冷却, 原子有足够时间找到能量最低的排列 (晶体)} \\
\text{如果冷却太快 (淬火 quenching), 原子被 "冻" 在能量较高的乱序状态 (局部最优)} \\ \quad \\
\text{对应关系:} \\
\begin{array}{|c|c|}
\hline
\text{物理} & \text{优化} \\
\hline
\text{系统状态} & \text{解 x} \\
\text{能量 E} & \text{目标函数 f(x)} \\
\text{温度 T} & \text{控制参数 (接受差解的程度)} \\
\text{基态 (最低能量)} & \text{全局最优解} \\
\hline
\end{array} \\ \quad \\
\text{和梯度下降 / 局部搜索的区别: 以一定概率接受更差的解, 从而能跳出局部最优}
$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from itertools import permutations
from scipy.optimize import minimize, dual_annealing

np.set_printoptions(linewidth=np.inf)
np.set_printoptions(suppress=True)

def plot_nicely():
    for ax in plt.gcf().get_axes():
        ax.grid(True, which="both", alpha=0.3)
        if ax.get_legend_handles_labels()[0]:
            ax.legend(fontsize='small')
    plt.tight_layout()

<a id="boltzmann"></a>

$$
\text{玻尔兹曼分布 (Boltzmann Distribution): 温度 T 下热平衡时, 系统处于状态 x 的概率} \\
\pi_T(x) = \frac{1}{Z_T} e^{-E(x) / T}, \quad Z_T = \sum_x e^{-E(x) / T} \ (\text{配分函数 Partition Function, 用来归一化}) \\ \quad \\
\text{1. 能量越低, 概率越大; 两个状态的概率之比只和能量差有关:} \\
\frac{\pi_T(x)}{\pi_T(y)} = e^{-(E(x) - E(y)) / T} \quad (Z_T \text{ 约掉了}) \\
\text{2. } T \to \infty \text{: } e^{-E/T} \to 1 \text{, 所有状态概率相同 (均匀分布, 随便乱走)} \\
\text{3. } T \to 0 \text{: 设全局最小能量为 } E^* \text{, 对任意 } E(x) > E^* \text{ 的状态} \\
\frac{\pi_T(x)}{\pi_T(x^*)} = e^{-(E(x) - E^*) / T} \to 0 \\
\text{所有概率都集中到全局最小值上} \implies \text{如果能从 } \pi_T \text{ 中抽样, 并让 T 慢慢降到 0, 就能找到全局最优}
$$

In [ ]:
# 一个离散的 "能量地形": 40 个状态, 有好几个局部最小值
states = np.arange(40)
energy = 3 + np.sin(states / 2.5) * 2 + 0.002 * (states - 30)**2 + 0.8 * np.cos(states * 1.3)
global_min = int(np.argmin(energy))
local_mins = [s for s in range(1, 39) if energy[s] < energy[s - 1] and energy[s] < energy[s + 1]]
print(f'局部最小值的位置 = {local_mins}, 全局最小值在状态 {global_min}, E* = {energy[global_min]:.4f}')

def boltzmann(E, T):
    w = np.exp(-(E - E.min()) / T)    # 减去最小值防止溢出, 比例不变
    return w / w.sum()

fig, axs = plt.subplots(1, 2, figsize=(8, 3))
axs[0].plot(states, energy, 'k.-')
axs[0].plot(global_min, energy[global_min], 'r*', markersize=12, label='global minimum')
axs[0].set_xlabel('state')
axs[0].set_ylabel('E')
for T in [10, 1, 0.3, 0.05]:
    axs[1].plot(states, boltzmann(energy, T), '.-', label=f'T = {T}')
    print(f'T = {T:<5}: 全局最小状态的概率 = {boltzmann(energy, T)[global_min]:.4f}')
axs[1].set_xlabel('state')
axs[1].set_ylabel('Boltzmann probability')
plot_nicely()
plt.show()

<a id="metropolis"></a>

$$
\text{Metropolis 准则: 怎样从 } \pi_T \text{ 中抽样 (不需要知道 } Z_T \text{)} \\
\text{当前状态 x, 随机提出一个邻居 y (提议分布对称: } q(y \mid x) = q(x \mid y) \text{)} \\
\text{接受概率: } A(x \to y) = \min\left(1, \ e^{-(E(y) - E(x)) / T}\right) \\
\text{更好的解 } (\Delta E \leq 0) \text{ 一定接受; 更差的解以概率 } e^{-\Delta E / T} \text{ 接受, 差得越多, 温度越低, 越不容易接受} \\ \quad \\
\text{为什么这样抽出来的就是 } \pi_T \text{? 这个过程是一个马尔可夫链, 转移概率 } P(x \to y) = q(y \mid x) A(x \to y) \\
\text{细致平衡 (Detailed Balance): } \pi(x) P(x \to y) = \pi(y) P(y \to x) \\
\text{验证: 不妨设 } E(y) \geq E(x) \text{ (反过来同理), 则 } A(x \to y) = e^{-(E(y) - E(x))/T}, \ A(y \to x) = 1 \\
\pi(x) P(x \to y) = \frac{e^{-E(x)/T}}{Z} \cdot q \cdot e^{-(E(y) - E(x))/T} = \frac{e^{-E(y)/T}}{Z} q \\
\pi(y) P(y \to x) = \frac{e^{-E(y)/T}}{Z} \cdot q \cdot 1 = \frac{e^{-E(y)/T}}{Z} q \quad \checkmark \\ \quad \\
\text{细致平衡} \implies \pi \text{ 是平稳分布 (Stationary Distribution): 一步转移之后分布不变} \\
\sum_x \pi(x) P(x \to y) = \sum_x \pi(y) P(y \to x) = \pi(y) \sum_x P(y \to x) = \pi(y) \\
\text{写成矩阵: } \pi^T P = \pi^T \text{, } \pi \text{ 是转移矩阵 } P^T \text{ 特征值 1 对应的特征向量} \\
\text{只要链是不可约且非周期的, 从任意初始状态出发, 分布都会收敛到 } \pi
$$

数学基础: [特征值与特征向量](../math_basics/linear_algebra.ipynb#eigen) · 相关模型: [马尔可夫链](markov_chain.ipynb)

In [ ]:
T_fix = 0.5
n_s = len(states)

# 提议: 从所有 n 个状态中等概率选一个 y (q(y|x) = 1/n, 对称)
# (如果只能走 ±1 步, 低温时会被能量壁垒困在一个局部最小值附近很久, 模拟要跑非常久才能接近平衡分布)
# 1. 精确的转移矩阵
P_mat = np.zeros((n_s, n_s))
for x in range(n_s):
    for y in range(n_s):
        if y != x:
            P_mat[x, y] = (1 / n_s) * min(1, np.exp(-(energy[y] - energy[x]) / T_fix))
    P_mat[x, x] = 1 - P_mat[x].sum()     # 被拒绝 (或提议到自己) 就留在原地
pi = boltzmann(energy, T_fix)
print(f'每一行和为 1? {np.allclose(P_mat.sum(axis=1), 1)}')
print(f'细致平衡 pi(x)P(x,y) == pi(y)P(y,x)? {np.allclose(pi[:, None] * P_mat, (pi[:, None] * P_mat).T)}')
print(f'平稳分布 pi^T P == pi^T ? {np.allclose(pi @ P_mat, pi)}')
eigval, eigvec = np.linalg.eig(P_mat.T)
v = np.real(eigvec[:, np.argmin(np.abs(eigval - 1))])
print(f'P^T 特征值 1 的特征向量 (归一化) == Boltzmann? {np.allclose(v / v.sum(), pi)}')

# 2. 模拟 Metropolis 链
rng = np.random.default_rng(0)
x = 0
visits = np.zeros(n_s)
for step in range(200000):
    y = rng.integers(n_s)
    if rng.random() < min(1, np.exp(-(energy[y] - energy[x]) / T_fix)):
        x = y
    visits[x] += 1
freq = visits / visits.sum()
print(f'模拟频率和 Boltzmann 的最大差 = {np.max(np.abs(freq - pi)):.4f}')

plt.figure(figsize=(4, 3))
plt.bar(states, freq, alpha=0.5, label='Metropolis visit frequency')
plt.plot(states, pi, 'r.-', label=f'Boltzmann T = {T_fix}')
plt.xlabel('state')
plot_nicely()
plt.show()

<a id="cooling"></a>

$$
\text{模拟退火 = 在逐渐降低的温度下跑 Metropolis} \\
\text{1. 初始解 } x_0 \text{, 初始温度 } T_0 \text{ (足够高, 一开始几乎什么都接受)} \\
\text{2. 每个温度下: 产生邻居 y, 按 Metropolis 准则接受或拒绝, 记录见过的最好解} \\
\text{3. 降温, 重复, 直到温度足够低} \\ \quad \\
\text{降温策略 (Cooling Schedule):} \\
\text{指数降温: } T_k = T_0 \alpha^k, \ \alpha \in [0.8, 0.99] \text{ (最常用)} \\
\text{对数降温: } T_k = \frac{c}{\ln(1 + k)} \text{ (理论上能保证以概率 1 收敛到全局最优, 但太慢)} \\
\text{线性降温: } T_k = T_0 - \beta k \\ \quad \\
\text{为什么要 "慢": 每个温度都要让链接近平衡分布 } \pi_T \text{, 降太快就和淬火一样, 卡在局部最优}
$$

In [ ]:
k = np.arange(1, 2001)
plt.figure(figsize=(4, 3))
plt.semilogy(k, 10 * 0.995**k, label='exponential 10 * 0.995^k')
plt.semilogy(k, 3 / np.log(1 + k), label='logarithmic 3 / ln(1 + k)')
plt.semilogy(k, np.clip(10 - 0.005 * k, 1e-3, None), label='linear 10 - 0.005k')
plt.xlabel('iteration k')
plt.ylabel('T')
plot_nicely()
plt.show()

# 在上面的离散地形上退火, 邻居 = 左右 3 步以内, 起点 = 状态 0
steps = np.array([-3, -2, -1, 1, 2, 3])
def anneal_discrete(T_schedule, rng):
    x = 0
    moves, us = rng.choice(steps, len(T_schedule)), rng.random(len(T_schedule))
    for T, m, u in zip(T_schedule, moves, us):
        y = x + m
        if 0 <= y < n_s and u < np.exp(-max(energy[y] - energy[x], 0) / T):
            x = y
    return x

rng = np.random.default_rng(1)
n_runs = 100
for name, sched in [('淬火 (T 每步减半)', np.maximum(10 * 0.5**np.arange(1, 3001), 1e-12)),
                    ('指数降温, 300 步从 10 降到 0.01', 10 * 1e-3**(np.arange(1, 301) / 300)),
                    ('指数降温, 3000 步从 10 降到 0.01', 10 * 1e-3**(np.arange(1, 3001) / 3000))]:
    ends = [anneal_discrete(sched, rng) for _ in range(n_runs)]
    print(f'{name:<28}: {n_runs} 次中结束在全局最小值的比例 = {np.mean(np.array(ends) == global_min):.2f}')
# 对数降温 c / ln(1 + k): c 小时一开始就降得很快 (k = 30 时已经 < c/3), 相当于淬火; c 大时又太慢, 实际中很少用

<a id="sa-continuous"></a>

$$
\text{例: 一元多峰函数的全局最小值} \\
f(x) = \frac{x^2}{20} + \sin(3x) + 0.5 \sin(7x), \quad x \in [-10, 10] \\
\text{有很多局部最小值, 从随机起点用局部优化 (梯度类方法) 通常只能找到离起点最近的那个} \\
\text{邻居的产生: } y = x + \varepsilon, \ \varepsilon \sim N(0, \sigma^2) \text{ (对称提议), 超出范围就截断回边界}
$$

In [ ]:
def f_multi(x):
    return x**2 / 20 + np.sin(3 * x) + 0.5 * np.sin(7 * x)

grid = np.linspace(-10, 10, 200001)
x_grid_best = grid[np.argmin(f_multi(grid))]
print(f'网格搜索的全局最小值: x = {x_grid_best:.5f}, f = {f_multi(x_grid_best):.6f}')

def simulated_annealing(f, x0, T0=5.0, alpha=0.995, n_iter=5000, step=1.0, bounds=(-10, 10), seed=0):
    rng = np.random.default_rng(seed)
    x, fx = x0, f(x0)
    best_x, best_f = x, fx
    T = T0
    history = []
    for k in range(n_iter):
        y = np.clip(x + rng.normal(0, step), *bounds)
        fy = f(y)
        if fy <= fx or rng.random() < np.exp(-(fy - fx) / T):    # Metropolis 准则
            x, fx = y, fy
            if fx < best_f:
                best_x, best_f = x, fx
        history.append((x, T))
        T *= alpha
    return best_x, best_f, np.array(history)

rng = np.random.default_rng(2)
starts = rng.uniform(-10, 10, 20)
local = [minimize(lambda v: f_multi(v[0]), [s]).x[0] for s in starts]
print(f'20 个随机起点做局部优化, 找到全局最小值的个数 = {np.sum(np.abs(np.array(local) - x_grid_best) < 1e-3)}')

sa_results = [simulated_annealing(f_multi, s, seed=i)[:2] for i, s in enumerate(starts)]
print(f'20 个随机起点做模拟退火, 找到全局最小值的个数 = {np.sum([abs(x - x_grid_best) < 0.05 for x, _ in sa_results])}')

x_sa, f_sa, hist = simulated_annealing(f_multi, 8.0)
res_da = dual_annealing(lambda v: f_multi(v[0]), bounds=[(-10, 10)], seed=0)
print(f'手写模拟退火 (起点 8): x = {x_sa:.5f}, f = {f_sa:.6f}')
print(f'scipy dual_annealing:   x = {res_da.x[0]:.5f}, f = {res_da.fun:.6f}')

fig, axs = plt.subplots(1, 2, figsize=(8, 3))
xx = np.linspace(-10, 10, 2000)
axs[0].plot(xx, f_multi(xx), 'k', linewidth=1)
axs[0].plot(x_sa, f_sa, 'r*', markersize=12, label='SA best')
axs[0].set_xlabel('x')
axs[0].set_ylabel('f(x)')
axs[1].plot(hist[:, 0], '.', markersize=1, label='current x')
axs[1].axhline(x_grid_best, color='r', linestyle='--', label='global minimum')
axs[1].set_xlabel('iteration (T decreasing)')
axs[1].set_ylabel('x')
plot_nicely()
plt.show()

<a id="tsp"></a>

$$
\text{例: 旅行商问题 (Traveling Salesman Problem, TSP)} \\
\text{n 个城市, 从一个城市出发, 每个城市恰好访问一次, 回到起点, 总距离最短} \\
\text{固定起点后, 不同的路线有 } \frac{(n - 1)!}{2} \text{ 种 (反方向走是同一条路线), n = 20 时约 } 6 \times 10^{16} \\ \quad \\
\text{邻居: 2-opt 变换, 选两个位置 i < j, 把中间这一段反转} \\
\text{原路线 } \cdots a \to b \to \cdots \to c \to d \cdots \text{ 变成 } \cdots a \to c \to \cdots \to b \to d \cdots \\
\text{只有两条边变了 (反转那一段内部的边长度不变, 只是方向反了), 所以能量变化可以 O(1) 计算:} \\
\Delta E = d(a, c) + d(b, d) - d(a, b) - d(c, d) \\
\text{(不用每次重新算整条路线的长度, 这是 TSP 上模拟退火能跑得快的关键)} \\
\text{几何意义: 如果路线有交叉, 2-opt 一定能把它解开并变短 (三角不等式)}
$$

In [ ]:
def tour_length(tour, Dm):
    return Dm[tour, np.roll(tour, -1)].sum()

def sa_tsp(Dm, n_iter=200000, T0=None, alpha=None, seed=0):
    rng = np.random.default_rng(seed)
    n = len(Dm)
    tour = np.arange(n)
    rng.shuffle(tour[1:])
    L = tour_length(tour, Dm)
    T0 = T0 or Dm.mean()
    alpha = alpha or (1e-4) ** (1 / n_iter)       # 从 T0 降到 T0 * 1e-4
    T = T0
    best, best_L = tour.copy(), L
    for _ in range(n_iter):
        i, j = sorted(rng.choice(np.arange(1, n), 2, replace=False))
        a, b, c, d = tour[i - 1], tour[i], tour[j], tour[(j + 1) % n]
        delta = Dm[a, c] + Dm[b, d] - Dm[a, b] - Dm[c, d]
        if delta <= 0 or rng.random() < np.exp(-delta / T):
            tour[i:j + 1] = tour[i:j + 1][::-1]
            L += delta
            if L < best_L - 1e-12:
                best, best_L = tour.copy(), L
        T *= alpha
    return best, best_L

# 1. 小规模: 和暴力枚举比较
rng = np.random.default_rng(3)
cities9 = rng.random((9, 2))
D9 = np.linalg.norm(cities9[:, None] - cities9[None, :], axis=2)
best_brute = min(((0,) + p for p in permutations(range(1, 9))), key=lambda t: tour_length(np.array(t), D9))
L_brute = tour_length(np.array(best_brute), D9)
tour9, L9 = sa_tsp(D9, n_iter=20000)
print(f'9 个城市: 暴力枚举 8! = 40320 条路线, 最短 = {L_brute:.6f}; 模拟退火 = {L9:.6f}, 一致? {np.isclose(L_brute, L9)}')
print(f'2-opt 的 O(1) 增量和直接重算的长度一致? {np.isclose(L9, tour_length(tour9, D9))}')

# 2. 40 个城市: 和最近邻贪心比较
cities = rng.random((40, 2))
D40 = np.linalg.norm(cities[:, None] - cities[None, :], axis=2)
nn_tour = [0]
for _ in range(39):
    rest = [c for c in range(40) if c not in nn_tour]
    nn_tour.append(min(rest, key=lambda c: D40[nn_tour[-1], c]))
nn_tour = np.array(nn_tour)
tour40, L40 = sa_tsp(D40)
print(f'40 个城市: 最近邻贪心 = {tour_length(nn_tour, D40):.4f}, 模拟退火 = {L40:.4f}')

fig, axs = plt.subplots(1, 2, figsize=(8, 3))
for ax, t, title in [(axs[0], nn_tour, f'nearest neighbor {tour_length(nn_tour, D40):.3f}'), (axs[1], tour40, f'simulated annealing {L40:.3f}')]:
    loop = np.append(t, t[0])
    ax.plot(cities[loop, 0], cities[loop, 1], 'b-', linewidth=1, label='tour')
    ax.plot(cities[:, 0], cities[:, 1], 'ro', markersize=3)
    ax.set_title(title)
    ax.set_aspect('equal')
plot_nicely()
plt.show()